# Tensor

In [ ]:
import torch
import numpy as np

## 张量初始化
rand 相关的初始化意味着每个元素都从均值为0、标准差为1的标准高斯分布中随机采样

In [ ]:
# type 1 : Directly from data
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

# type 2 : From a NumPy array
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

# type 3 : From another tensor
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

print(f"Origin Tensor: \n {x_data} \n")
print(f"Shape : {x_data.shape}")

## 随机或常量初始化

In [ ]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)
arange_tensor = torch.arange(6)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")
print(f"Arange Tensor: \n {arange_tensor}")

arange_tensor = arange_tensor.reshape(shape)
print(f"Arange Tensor: \n {arange_tensor}")

## 张量属性

In [ ]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")
print(f"num of all elements: {tensor.numel()}")

## 张量运算

In [ ]:
tensor = torch.ones(2, 3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# tensor = tensor.to(device)
print(f"Device tensor is stored on: {tensor.device}")
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

### 张量拼接

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1) # 对于row和col的第二维拼接
t2 = torch.cat([tensor, tensor], dim=0) # 对于row和col的第一维拼接
print(t1)
print(t2)

### 算数运算

In [ ]:
# 四种等价矩阵-向量积
v = torch.rand(4)
x1 = tensor.matmul(v)
x2 = tensor.mv(v)
print(x2)
x3 = tensor @ v
print(x3)
x4 = torch.rand_like(x1)
torch.matmul(tensor, v, out=x4)
print(x4)

# 四种种等价的矩阵乘法
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)
y3 = tensor.mm(tensor.T)

y4 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y4)
print(y4)

# 三种等价的逐元素相乘
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)
print(z3)

# 逐元素求和
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

# 原地操作
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)
tensor.copy_(3)
print(tensor)
# 原地操作节省了一些内存，但在计算导数时可能存在问题，因为历史会立即丢失。因此，不建议使用

x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
print(x + y)
print(x - y)
print(x * y)
print(x / y)
print(x ** y) # **运算符是求幂运算

x_exp = torch.exp(x)
print(x_exp) # 指数函数运算


# 逻辑运算符，对于每个位置，如果X和Y在该位置满足条件，则新张量中相应项的值为1
print(x == y)
print(x > y)
print(x < y)

## 批量矩阵乘法

In [ ]:
X = torch.ones((2, 1, 4))
Y = torch.ones((2, 4, 6))
torch.bmm(X, Y).shape

## 索引和切片
与任何Python数组一样：第一个元素的索引是0，最后一个元素索引是-1； 可以指定范围以包含第一个元素和最后一个之前的元素

In [ ]:
x = torch.arange(6).reshape((3, 2))
# x = torch.arange(6).reshape(3, 2)
# 上面的两种reshape函数用法都可以

print(x)
print(x[-1])
print(x[1:3])
x[1, 1] = 9
print(x)
x[0:2, :] = 12
print(x)

## 原地操作

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

# 运行一些操作可能会导致为新结果分配内存，比如
before = id(Y)
Y = Y + X
print(id(Y) == before)

# 可以使用Z[:]的方式来实现原地操作
Z = torch.zeros_like(Y)
print('id(Z):', id(Z))
Z[:] = X + Y
print('id(Z):', id(Z))

# 同样也可以用 += 的方式实现
before = id(X)
X += Y
print(id(X) == before)

## 广播机制
广播机制规则：
如果遵守以下规则，则两个tensor是“可广播的”：
1. 每个tensor至少有一个维度；
2. 遍历tensor所有维度时，从末尾随开始遍历，两个tensor存在下列情况：
  - tensor维度相等。
  - tensor维度不等且其中一个维度为1。
  - tensor维度不等且其中一个维度不存在。

如果两个tensor是“可广播的”，则计算过程遵循下列规则：
1. 如果两个tensor的维度不同，则在维度较小的tensor的前面增加维度，使它们维度相等。
2. 对于每个维度，计算结果的维度值取两个tensor中较大的那个值。
3. 两个tensor扩展维度的过程是将数值进行复制。

In [ ]:
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b

In [ ]:
# 如果让它们相加，它们的形状不匹配。 会自动将两个矩阵广播为一个更大的 3x2 矩阵，再进行相加
a + b

## 范数

In [ ]:
# 向量L2范数，向量元素平方和的平方根
u = torch.tensor([3.0, -4.0])
torch.norm(u)

In [ ]:
# 向量L1范数，向量元素的绝对值之和
torch.abs(u).sum()

In [ ]:
# 矩阵F范数，矩阵元素平方和的平方根
torch.norm(torch.ones((4, 9)))

## 降维

In [ ]:
x = torch.arange(4, dtype=torch.float32)
x, x.sum()

In [ ]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
print(A)

# 指定张量沿哪一个轴来通过求和降低维度
# axis=0意味着通过求和所有行的元素来降维，即把行消除了，按列求和
A_sum_axis0 = A.sum(axis=0)
A_sum_axis0, A_sum_axis0.shape

In [ ]:
# 同理，axis=1意味着把列消除了，按行求和
A_sum_axis1 = A.sum(axis=1)
A_sum_axis1, A_sum_axis1.shape

In [ ]:
# mean 函数同样可以用于某一个维度
A.mean(axis=0), A.sum(axis=0), A.shape[0], A.sum(axis=0) / A.shape[0]

In [ ]:
# 上面使用sum函数之后，A从二维矩阵变成了一维向量，其实可以保持轴数不变
sum_A = A.sum(axis=1, keepdims=True)
sum_A

In [ ]:
# 则后续可以通过广播将A除以sum_A
A / sum_A

In [ ]:
# 想沿某个轴计算A元素的累积总和， 比如axis=0（按行计算），可以调用cumsum函数。 此函数不会沿任何轴降低输入张量的维度
A.cumsum(axis=0)

## 与 NumPy 的转换
numpy() 和 from_numpy() 方法是可以共享内存的，两者的变化是会相互影响的
但如果使用torch.tensor(ndarray)由numpy数组生成tensor，就不会共享内存。

In [ ]:
# Tensor to NumPy array
t = torch.ones(3)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")
print(type(t), type(n))

t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")
# 这里两者的变化会相互影响

In [ ]:
# NumPy array to Tensor
n = np.ones(5)
t = torch.from_numpy(n)
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

# 这里两者的变化同样会相互影响